In [16]:
# import relevant packages
import capytaine as cpt
import vtk
import matplotlib
from numpy import inf
import numpy as np

In [6]:
cpt.set_logging('INFO') # gives more details on what capytaine is doing. other modes are 'WARNING', 'DEBUG'

In [7]:
# loading a mesh

sphere = cpt.mesh_sphere(radius=1.0, center=(0,0,-2), name='my sphere')

# note we can import premade meshes i think from somewhre in the documentation which could be useful for a buoy

In [ ]:
# visualise mesh

sphere.show() # this is so sick


In [11]:
# defining a body 

'''
before solving a diffraction or radiaiton problem we need to defin the degreess of freedom of out body. 
We will start with heave only but following this tutorial I will put what they put
'''

body = cpt.FloatingBody(mesh=sphere, 
                        dofs = cpt.rigid_body_dofs(rotation_center=(0,0,-2)),
                        center_of_mass=(0,0,-2))
         

[15:09:08] INFO     The rotation dof Roll has been initialized around the point: FloatingBody(..., name="my        
                    sphere").rotation_center = [ 0.  0. -2.]

           INFO     The rotation dof Pitch has been initialized around the point: FloatingBody(..., name="my       
                    sphere").rotation_center = [ 0.  0. -2.]

           INFO     The rotation dof Yaw has been initialized around the point: FloatingBody(..., name="my         
                    sphere").rotation_center = [ 0.  0. -2.]

           INFO     New floating body: FloatingBody(mesh=Mesh(..., name="my sphere"), lid_mesh=None, dofs={"Surge":
                    ..., "Sway": ..., "Heave": ..., "Roll": ..., "Pitch": ..., "Yaw": ...}, center_of_mass=[ 0.  0.
                    -2.], name="my sphere").

In [12]:
# printing dofs dictionary. 

print(body.dofs.keys())

dict_keys(['Surge', 'Sway', 'Heave', 'Roll', 'Pitch', 'Yaw'])


In [13]:
'''
Capytain can directly perform some hydrostatic computations. 
You can get parameters such as volume, wet surface area, waterplane area, 
center of buoyancy, metcentric radius, 
height, hydrostatic stiffness and intertia matrix

'''

hydrostatics = body.compute_hydrostatics(rho=1025.0)

print(hydrostatics["disp_volume"])
# 3.82267415555807

print(hydrostatics["hydrostatic_stiffness"])
# <xarray.DataArray 'hydrostatic_stiffness' (influenced_dof: 6, radiating_dof: 6)> Size: 288B
# [...]
# Coordinates:
#   * influenced_dof  (influenced_dof) <U5 120B 'Surge' 'Sway' ... 'Pitch' 'Yaw'
#   * radiating_dof   (radiating_dof) <U5 120B 'Surge' 'Sway' ... 'Pitch' 'Yaw'

print(hydrostatics["inertia_matrix"])
# <xarray.DataArray 'inertia_matrix' (influenced_dof: 6, radiating_dof: 6)> Size: 288B
# [...]
# Coordinates:
#   * influenced_dof  (influenced_dof) <U5 120B 'Surge' 'Sway' ... 'Pitch' 'Yaw'
#   * radiating_dof   (radiating_dof) <U5 120B 'Surge' 'Sway' ... 'Pitch' 'Yaw'

[15:11:58] INFO     Clipping my sphere with respect to Plane(normal=[0. 0. 1.], point=[0. 0. 0.])

           INFO     Clipping my sphere by Plane(normal=[0. 0. 1.], point=[0. 0. 0.]): no action.

3.82267415555807
<xarray.DataArray 'hydrostatic_stiffness' (influenced_dof: 6, radiating_dof: 6)> Size: 288B
array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00, -5.93064660e-13,
         4.22994647e-13, -1.02914162e-12,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  4.22994647e-13,
        -1.70670146e-11, -1.89148380e-13,  1.86749956e-12],
       [ 0.00000000e+00,  0.00000000e+00, -1.02914162e-12,
        -1.89148380e-13, -1.68834534e-11, -9.94289528e-14],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]])
Coordinates:
  * influenced_dof  (influenced_dof) <U5 120B 'Surge' 'Sway' ... 'Pitch' 'Yaw'
  * radiating_dof   (radiating_dof) <U5 120B 'Surge' 'Sway' ... 'Pitch' 'Yaw'
<xarra

In [15]:
# defining linear flow problem

problem = cpt.RadiationProblem(body=body, radiating_dof="Heave", omega=1.0,
                                water_depth=inf, g=9.81, rho=1000)

# note** can define wavelength or wavenumber instead of omega

# more parameters are automatically compute such as

print(problem.wavenumber)
# 0.10471975511965977
print(problem.period)
# 6.199134450374511


0.1019367991845056
6.283185307179586


In [17]:
'''
Capytaine also implement a DiffractionPriblem class which does not take a readiating_dof argument 
but instad requires a wabe_direction in radians:
'''

diffraction_problem = cpt.DiffractionProblem(body=body, wave_direction=np.pi/2, omega=1.0)

SOLVING THEM PROBLEM

In [18]:
# intialise the BEM (boundary element method) solver

solver = cpt.BEMSolver()

[15:17:43] WARNING  Precomputing tabulation, it may take a few seconds.

In [19]:
# create  result object to stor results

result = solver.solve(problem) # solves the problem we defined earlier (radiation problem)

# .solve() method returns a result object. This containts all data from problem as well as output data

# output data include the added mass and  radiation damping



[15:20:39] INFO     Solve RadiationProblem(body=FloatingBody(..., name="my sphere"), omega=1.000, water_depth=inf, 
                    radiating_dof='Heave').

In [20]:
print(result.added_masses)

print(result.radiation_dampings)

{'Surge': 2.6290081223123707e-13, 'Sway': 2.8421709430404007e-13, 'Heave': 2207.4220994883012, 'Roll': -2.842170943040401e-14, 'Pitch': 1.2534417948018017e-13, 'Yaw': -1.1455632523845817e-14}
{'Surge': -7.105427357601002e-15, 'Sway': 0.0, 'Heave': 13.620465603186025, 'Roll': 4.829470157119431e-15, 'Pitch': -6.938893903907228e-17, 'Yaw': -2.2504402269422007e-15}


In [22]:
diffraction_result = solver.solve(diffraction_problem)
print(diffraction_result.forces)
# {'Surge': np.complex128(2.5934809855243657e-13+2.2870594307278225e-13j),
#  'Sway': np.complex128(5.969301397957423-1928.0584773706814j),
#  'Heave': np.complex128(-1802.2378814572921-10.9509664655968j),
#  'Roll': np.complex128(-0.010423009597921862+4.185948400947856j),
#  'Pitch': np.complex128(-3.319566843629218e-14+2.0039525594484076e-14j),
#  'Yaw': np.complex128(-6.444497858876913e-15+5.167800131833467e-14j)
#  }

[15:23:25] INFO     Solve DiffractionProblem(body=FloatingBody(..., name="my sphere"), omega=1.000,                
                    water_depth=inf, wave_direction=1.571).

{'Surge': np.complex128(-1.5276668818842154e-13+3.6992631180510216e-13j), 'Sway': np.complex128(5.96912291490959-1927.987781471024j), 'Heave': np.complex128(-1802.4250845364154-10.956460558566988j), 'Roll': np.complex128(-0.010419967639343497+4.183607023598109j), 'Pitch': np.complex128(-1.0363931934875836e-13+3.58046925441613e-15j), 'Yaw': np.complex128(1.0563772609203808e-14+6.089959210929401e-14j)}


In [21]:
# gather results in arrays

all_radiation_problems = [cpt.RadiationProblem(body=body, radiating_dof=dof, omega=1.0) for dof in body.dofs]
all_radiation_results = solver.solve_all(all_radiation_problems)

c:\Users\edwar\anaconda3\envs\capytaine_env\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[15:21:33] INFO     Solve RadiationProblem(body=FloatingBody(..., name="my sphere"), omega=1.000, water_depth=inf, 
                    radiating_dof='Surge').

           INFO     Solve RadiationProblem(body=FloatingBody(..., name="my sphere"), omega=1.000, water_depth=inf, 
                    radiating_dof='Sway').

           INFO     Solve RadiationProblem(body=FloatingBody(..., name="my sphere"), omega=1.000, water_depth=inf, 
                    radiating_dof='Heave').

           INFO     Solve RadiationProblem(body=FloatingBody(..., name="my sphere"), omega=1.000, water_depth=inf, 
                    radiating_dof='Roll').

           INFO     Solve RadiationProblem(body=FloatingBody(..., name="my sphere"), omega=1.000, water_depth=inf, 
                    radiating_dof='Pitch').

           INFO     Solve RadiationProblem(body=FloatingBody(..., name="my sphere"), omega=1.000, water_depth=inf, 
                    radiating_dof='Yaw').

           INFO     Solver timer summary:                                                                          
                                         total  nb_calls      mean                                                 
                    task                                                                                           
                    Solve total       0.067860         7  0.009694                                                 
                      Green function  0.010709         7  0.001530                                                 
                      Linear solver   0.017413         7  0.002488

In [23]:
# results can be gathered together as follows

dataset = cpt.assemble_dataset([diffraction_result] + all_radiation_results)

In [24]:
# the data is being stored in a clever way it can be accessed like this
dataset['added_mass'].sel(radiating_dof=["Surge", "Heave"], influenced_dof=["Surge", "Heave"], omega=1.0)


<xarray.DataArray 'added_mass' (radiating_dof: 2, influenced_dof: 2)> Size: 32B
array([[2.36165669e+03, 4.61852778e-14],
       [2.62900812e-13, 2.20742210e+03]])
Coordinates:
  * radiating_dof   (radiating_dof) category 50B Surge Heave
  * influenced_dof  (influenced_dof) category 50B Surge Heave
    g               float64 8B 9.81
    rho             float64 8B 1e+03
    body_name       StringDType(na_object=nan) 16B 'my sphere'
    water_depth     float64 8B inf
    forward_speed   float64 8B 0.0
    omega           float64 8B 1.0
    freq            float64 8B 0.1592
    period          float64 8B 6.283
    wavenumber      float64 8B 0.1019
    wavelength      float64 8B 61.64
Attributes:
    long_name:  Added mass

Post-Processing and collection in dataset